In [1]:
from pathlib import Path

import pandas as pd
import sklearn

PROJECT_ROOT = Path.cwd().parent

assert (PROJECT_ROOT / "PROJECT_PLAN.md").exists(), (
    "Make sure this notebook is saved in the project's notebooks directory"
)

print("scikit-learn:", sklearn.__version__)

scikit-learn: 1.9.1


In [2]:
training_text = pd.read_parquet(
    PROJECT_ROOT / "data/processed/credit_card_training_text.parquet"
)

quality_flags = pd.read_parquet(
    PROJECT_ROOT / "data/processed/training_text_quality_flags.parquet"
)

flag_columns = [
    "complaint_id",
    "text_group_id",
    "text_group_size",
    "has_label_conflict",
]

review_records = training_text.merge(
    quality_flags[flag_columns],
    on="complaint_id",
    how="left",
    validate="one_to_one",
    indicator=True,
)

assert len(review_records) == 33117
assert review_records["_merge"].eq("both").all(), (
    "Some text records do not have matching quality flags"
)

review_records = review_records.drop(columns="_merge")

print("Text records with matched quality flags:", len(review_records))

Text records with matched quality flags: 33117


In [3]:
# Select the earliest record in each exact normalized text group
representatives = (
    review_records
    .sort_values(
        ["received_date", "complaint_id"],
        kind="stable",
    )
    .drop_duplicates(
        subset="text_group_id",
        keep="first",
    )
    .reset_index(drop=True)
)

# The selected record's label does not resolve group-level label conflicts
representatives = representatives.rename(
    columns={"issue": "representative_issue"}
)

assert representatives["text_group_id"].is_unique
assert len(representatives) == 30257
assert representatives["has_label_conflict"].notna().all()

representative_summary = pd.DataFrame([{
    "original_text_rows": len(review_records),
    "representative_text_rows": len(representatives),
    "conflicting_label_groups_retained": int(
        representatives["has_label_conflict"].sum()
    ),
}])

display(representative_summary)

,original_text_rows,representative_text_rows,conflicting_label_groups_retained
0,33117,30257,77


In [4]:
REPRESENTATIVES_PATH = (
    PROJECT_ROOT
    / "data/processed/training_text_representatives.parquet"
)

representatives.to_parquet(
    REPRESENTATIVES_PATH,
    index=False,
    compression="zstd",
)

representative_summary.to_csv(
    PROJECT_ROOT / "reports/near_duplicate_input_summary.csv",
    index=False,
)

print("Saved:", REPRESENTATIVES_PATH.relative_to(PROJECT_ROOT))

Saved: data/processed/training_text_representatives.parquet


In [5]:
import json
from time import perf_counter

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

# Normalize whitespace for similarity search without changing source text
search_text = (
    representatives["narrative"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    max_features=50000,
    sublinear_tf=True,
    norm="l2",
    dtype=np.float32,
)

started_at = perf_counter()
text_vectors = vectorizer.fit_transform(search_text)
vectorization_seconds = perf_counter() - started_at

print("Vectorization completed.")
print("Elapsed seconds:", round(vectorization_seconds, 2))
print("Matrix shape:", text_vectors.shape)

Vectorization completed.
Elapsed seconds: 7.43
Matrix shape: (30257, 50000)


In [6]:
nonzero_features_per_row = np.asarray(
    text_vectors.getnnz(axis=1)
).ravel()

zero_vector_mask = nonzero_features_per_row == 0

matrix_storage_bytes = (
    text_vectors.data.nbytes
    + text_vectors.indices.nbytes
    + text_vectors.indptr.nbytes
)

vector_summary = pd.DataFrame([{
    "text_rows": text_vectors.shape[0],
    "features": text_vectors.shape[1],
    "nonzero_entries": text_vectors.nnz,
    "zero_vector_rows": int(zero_vector_mask.sum()),
    "sparse_matrix_storage_mb": round(
        matrix_storage_bytes / 1_000_000, 2
    ),
    "vectorization_seconds": round(vectorization_seconds, 2),
}])

display(vector_summary)

assert text_vectors.shape[0] == len(representatives)

representatives["has_search_features"] = ~zero_vector_mask

,text_rows,features,nonzero_entries,zero_vector_rows,sparse_matrix_storage_mb,vectorization_seconds
0,30257,50000,6814956,0,54.64,7.43


In [7]:
vector_config = {
    "purpose": "Training-only lexical near-duplicate candidate search",
    "sklearn_version": sklearn.__version__,
    "analyzer": "word",
    "lowercase": True,
    "ngram_range": [1, 2],
    "min_df": 2,
    "max_df": 0.95,
    "max_features": 50000,
    "sublinear_tf": True,
    "norm": "l2",
    "dtype": "float32",
    "input_rows": len(representatives),
}

(PROJECT_ROOT / "configs/near_duplicate_vectorizer.json").write_text(
    json.dumps(vector_config, indent=2) + "\n",
    encoding="utf-8",
)

vector_summary.to_csv(
    PROJECT_ROOT / "reports/near_duplicate_vector_summary.csv",
    index=False,
)

print("Vectorizer configuration and summary saved.")

Vectorizer configuration and summary saved.


In [8]:
from sklearn.neighbors import NearestNeighbors

TOP_K = 5
PILOT_SIZE = min(128, text_vectors.shape[0])
RANDOM_SEED = 42

neighbor_search = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_neighbors=TOP_K + 1,
    n_jobs=1,
)

neighbor_search.fit(text_vectors)

rng = np.random.default_rng(RANDOM_SEED)
pilot_indices = np.sort(
    rng.choice(
        text_vectors.shape[0],
        size=PILOT_SIZE,
        replace=False,
    )
)

print("Reference texts:", text_vectors.shape[0])
print("Pilot query texts:", len(pilot_indices))
print("Target neighbors per query:", TOP_K)

Reference texts: 30257
Pilot query texts: 128
Target neighbors per query: 5


In [9]:
started_at = perf_counter()

# Limit the working-memory target for pairwise distance calculations
with sklearn.config_context(working_memory=128):
    pilot_distances, pilot_neighbors = neighbor_search.kneighbors(
        text_vectors[pilot_indices]
    )

pilot_seconds = perf_counter() - started_at

best_neighbor_scores = []

for query_index, distances, neighbors in zip(
    pilot_indices,
    pilot_distances,
    pilot_neighbors,
):
    other_scores = [
        float(np.clip(1.0 - distance, 0.0, 1.0))
        for distance, neighbor_index in zip(distances, neighbors)
        if neighbor_index != query_index
    ]

    best_neighbor_scores.append(max(other_scores))

best_neighbor_scores = np.asarray(best_neighbor_scores)

rough_full_search_minutes = (
    pilot_seconds
    * text_vectors.shape[0]
    / PILOT_SIZE
    / 60
)

pilot_summary = pd.DataFrame([{
    "reference_texts": text_vectors.shape[0],
    "pilot_queries": PILOT_SIZE,
    "pilot_seconds": round(pilot_seconds, 2),
    "median_best_neighbor_similarity": round(
        float(np.median(best_neighbor_scores)), 4
    ),
    "queries_with_best_similarity_ge_090": int(
        (best_neighbor_scores >= 0.90).sum()
    ),
    "rough_full_search_minutes": round(
        rough_full_search_minutes, 2
    ),
}])

display(pilot_summary)

,reference_texts,pilot_queries,pilot_seconds,median_best_neighbor_similarity,queries_with_best_similarity_ge_090,rough_full_search_minutes
0,30257,128,0.2,0.1776,14,0.8


In [10]:
pilot_summary.to_csv(
    PROJECT_ROOT / "reports/near_duplicate_search_pilot.csv",
    index=False,
)

print("Pilot search summary saved.")

Pilot search summary saved.


In [11]:
BATCH_SIZE = 128
CANDIDATE_THRESHOLD = 0.90

candidate_scores = {}
started_at = perf_counter()
total_queries = text_vectors.shape[0]

for start in range(0, total_queries, BATCH_SIZE):
    end = min(start + BATCH_SIZE, total_queries)

    with sklearn.config_context(working_memory=128):
        batch_distances, batch_neighbors = (
            neighbor_search.kneighbors(text_vectors[start:end])
        )

    for query_index, distances, neighbors in zip(
        range(start, end),
        batch_distances,
        batch_neighbors,
    ):
        valid_neighbors = [
            (
                int(neighbor_index),
                float(np.clip(1.0 - distance, 0.0, 1.0)),
            )
            for distance, neighbor_index in zip(distances, neighbors)
            if neighbor_index != query_index
        ][:TOP_K]

        for neighbor_index, similarity in valid_neighbors:
            if similarity < CANDIDATE_THRESHOLD:
                continue

            # Store A-B and B-A as the same unordered pair
            pair = tuple(sorted((query_index, neighbor_index)))

            candidate_scores[pair] = max(
                similarity,
                candidate_scores.get(pair, 0.0),
            )

    if start % (BATCH_SIZE * 10) == 0 or end == total_queries:
        print(f"Processed {end:,} / {total_queries:,} queries")

search_seconds = perf_counter() - started_at

print("Search completed.")
print("Elapsed seconds:", round(search_seconds, 2))
print("Unique candidate pairs:", len(candidate_scores))

Processed 128 / 30,257 queries
Processed 1,408 / 30,257 queries
Processed 2,688 / 30,257 queries
Processed 3,968 / 30,257 queries
Processed 5,248 / 30,257 queries
Processed 6,528 / 30,257 queries
Processed 7,808 / 30,257 queries
Processed 9,088 / 30,257 queries
Processed 10,368 / 30,257 queries
Processed 11,648 / 30,257 queries
Processed 12,928 / 30,257 queries
Processed 14,208 / 30,257 queries
Processed 15,488 / 30,257 queries
Processed 16,768 / 30,257 queries
Processed 18,048 / 30,257 queries
Processed 19,328 / 30,257 queries
Processed 20,608 / 30,257 queries
Processed 21,888 / 30,257 queries
Processed 23,168 / 30,257 queries
Processed 24,448 / 30,257 queries
Processed 25,728 / 30,257 queries
Processed 27,008 / 30,257 queries
Processed 28,288 / 30,257 queries
Processed 29,568 / 30,257 queries
Processed 30,257 / 30,257 queries
Search completed.
Elapsed seconds: 45.9
Unique candidate pairs: 2942


In [12]:
candidate_pairs = pd.DataFrame(
    [
        {
            "left_index": left,
            "right_index": right,
            "cosine_similarity": score,
        }
        for (left, right), score in candidate_scores.items()
    ],
    columns=["left_index", "right_index", "cosine_similarity"],
)

metadata_columns = [
    "text_group_id",
    "complaint_id",
    "representative_issue",
    "has_label_conflict",
]

text_lengths = representatives["narrative"].str.len()

for side in ["left", "right"]:
    indices = candidate_pairs[f"{side}_index"]

    for column in metadata_columns:
        candidate_pairs[f"{side}_{column}"] = indices.map(
            representatives[column]
        )

    candidate_pairs[f"{side}_chars"] = indices.map(text_lengths)

candidate_pairs = candidate_pairs.sort_values(
    ["cosine_similarity", "left_index", "right_index"],
    ascending=[False, True, True],
).reset_index(drop=True)

print("Candidate pairs prepared:", len(candidate_pairs))

Candidate pairs prepared: 2942


In [13]:
candidate_summary = pd.DataFrame([{
    "query_texts": total_queries,
    "top_k": TOP_K,
    "batch_size": BATCH_SIZE,
    "candidate_threshold": CANDIDATE_THRESHOLD,
    "unique_candidate_pairs": len(candidate_pairs),
    "text_groups_in_candidate_pairs": pd.concat(
        [
            candidate_pairs["left_text_group_id"],
            candidate_pairs["right_text_group_id"],
        ],
        ignore_index=True,
    ).nunique(),
    "pairs_with_similarity_ge_095": int(
        (candidate_pairs["cosine_similarity"] >= 0.95).sum()
    ),
    "pairs_with_similarity_ge_099": int(
        (candidate_pairs["cosine_similarity"] >= 0.99).sum()
    ),
    "search_seconds": round(search_seconds, 2),
}])

display(candidate_summary)

candidate_pairs.to_parquet(
    PROJECT_ROOT / "data/processed/near_duplicate_candidates.parquet",
    index=False,
)

candidate_summary.to_csv(
    PROJECT_ROOT / "reports/near_duplicate_candidate_summary.csv",
    index=False,
)

CONFIG_PATH = PROJECT_ROOT / "configs/near_duplicate_vectorizer.json"
config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))

config["candidate_search"] = {
    "metric": "cosine",
    "algorithm": "brute",
    "top_k_non_self": TOP_K,
    "batch_size": BATCH_SIZE,
    "threshold": CANDIDATE_THRESHOLD,
    "working_memory_mb": 128,
    "n_jobs": 1,
}

CONFIG_PATH.write_text(
    json.dumps(config, indent=2) + "\n",
    encoding="utf-8",
)

print("Candidate pairs, summary, and search configuration saved.")

,query_texts,top_k,batch_size,candidate_threshold,unique_candidate_pairs,text_groups_in_candidate_pairs,pairs_with_similarity_ge_095,pairs_with_similarity_ge_099,search_seconds
0,30257,5,128,0.9,2942,1704,1994,350,45.9


Candidate pairs, summary, and search configuration saved.


In [14]:
REVIEW_PATH = (
    PROJECT_ROOT / "data/processed/near_duplicate_review.csv"
)

if not REVIEW_PATH.exists():
    candidates_for_review = candidate_pairs.copy()

    candidates_for_review["similarity_band"] = pd.cut(
        candidates_for_review["cosine_similarity"],
        bins=[0.90, 0.95, 0.99, 1.000001],
        labels=["0.90-0.95", "0.95-0.99", "0.99-1.00"],
        right=False,
    )

    sampled_bands = []

    for band, group in candidates_for_review.groupby(
        "similarity_band",
        observed=True,
    ):
        sampled = group.sample(
            n=min(10, len(group)),
            random_state=42,
        )
        sampled_bands.append(sampled)

    review = pd.concat(
        sampled_bands,
        ignore_index=True,
    )

    review = review[
        [
            "left_text_group_id",
            "right_text_group_id",
            "cosine_similarity",
            "similarity_band",
            "left_chars",
            "right_chars",
        ]
    ].copy()

    review.insert(
        0,
        "review_id",
        [f"ND{i:03d}" for i in range(1, len(review) + 1)],
    )

    review["review_label"] = ""
    review["review_notes"] = ""

    review.to_csv(REVIEW_PATH, index=False)
    print("Review sample created.")
else:
    # Preserve existing manual annotations when the cell is rerun
    review = pd.read_csv(
        REVIEW_PATH,
        keep_default_na=False,
    )
    print("Existing review sample loaded.")

print("Review pairs:", len(review))
print(
    review.groupby("similarity_band", observed=True)
    .size()
    .to_string()
)

Existing review sample loaded.
Review pairs: 30
similarity_band
0.90-0.95    10
0.95-0.99    10
0.99-1.00    10


In [15]:
text_by_group = representatives.set_index(
    "text_group_id"
)["narrative"]

first_review_cases = (
    review.groupby("similarity_band", observed=True, sort=True)
    .head(1)
)

PREVIEW_CHARS = 1500

for case in first_review_cases.itertuples(index=False):
    left_text = text_by_group.loc[case.left_text_group_id]
    right_text = text_by_group.loc[case.right_text_group_id]

    print("Review ID:", case.review_id)

    print("\nTEXT A:")
    print(left_text[:PREVIEW_CHARS])
    print("Preview truncated:", len(left_text) > PREVIEW_CHARS)

    print("\nTEXT B:")
    print(right_text[:PREVIEW_CHARS])
    print("Preview truncated:", len(right_text) > PREVIEW_CHARS)

    print("\n" + "=" * 80 + "\n")

Review ID: ND001

TEXT A:
This is a formal complain against Macy 's. It all started with my identity was stolen and used to create a credit card account with Macy's/Citibank in XX/XX/XXXX. I learned about the existence of the account when I received a bill in XX/XX/XXXX. I immediately reported fraud to Macy 's. How or why would someone has my information, despite incomplete or incorrect, was allowed to create an account is beyond me. Additionally, when I made the first phone call in XX/XX/XXXX, I asked them to do whatever it takes to resolved them. Macy 's action was to closed the account while doing their investigation. HOWEVER, it was later reported on my credit report that this account was " closed by the customer '' when it was closed by Macy 's due to the fraud investigation. I have made followed-up phone calls with Macy 's to asked and confirmed that the account was closed, in XXXX XXXX. During these phone calls, Macy 's representative repeatedly asked for my account number when 

In [16]:
review = pd.read_csv(
    REVIEW_PATH,
    keep_default_na=False,
)

annotations = {
    "ND001": {
        "label": "uncertain",
        "notes": (
            "Visible prefixes strongly overlap, but both previews are "
            "truncated. The remaining text must be checked before classification."
        ),
    },
    "ND011": {
        "label": "shared_template",
        "notes": (
            "The texts share generic complaint wording. The account descriptor "
            "differs or is redacted, and specific event details are insufficient "
            "to establish equivalence."
        ),
    },
    "ND021": {
        "label": "near_duplicate",
        "notes": (
            "The full narratives match apart from an account-name redaction. "
            "The stated amounts, problem description, and requested actions match. "
            "This does not establish that the underlying events are identical."
        ),
    },
}

assert review["review_id"].is_unique

for review_id, annotation in annotations.items():
    mask = review["review_id"].eq(review_id)

    if mask.sum() != 1:
        raise ValueError(f"Expected one review row for {review_id}")

    review.loc[mask, "review_label"] = annotation["label"]
    review.loc[mask, "review_notes"] = annotation["notes"]

review.to_csv(REVIEW_PATH, index=False)

print("Saved annotations:", ", ".join(annotations))

Saved annotations: ND001, ND011, ND021


In [17]:
from difflib import SequenceMatcher

case = review.loc[review["review_id"].eq("ND001")].iloc[0]

left_text = text_by_group.loc[case["left_text_group_id"]]
right_text = text_by_group.loc[case["right_text_group_id"]]

left_words = left_text.split()
right_words = right_text.split()

matcher = SequenceMatcher(
    None,
    left_words,
    right_words,
    autojunk=False,
)

print("Review ID: ND001")
print("Text A characters:", len(left_text))
print("Text B characters:", len(right_text))

difference_count = 0

for operation, a_start, a_end, b_start, b_end in matcher.get_opcodes():
    if operation == "equal":
        continue

    difference_count += 1

    print(f"\nDifference {difference_count}: {operation}")
    print(
        "A:",
        " ".join(
            left_words[
                max(0, a_start - 8):min(len(left_words), a_end + 8)
            ]
        ),
    )
    print(
        "B:",
        " ".join(
            right_words[
                max(0, b_start - 8):min(len(right_words), b_end + 8)
            ]
        ),
    )

print("\nDiffering sections:", difference_count)

Review ID: ND001
Text A characters: 4442
Text B characters: 5400

Difference 1: replace
A: This is a formal complain against Macy 's. It all started with my identity was stolen
B: This is a formal complain against Macy's/Citi bank XXXX It all started with my identity was stolen

Difference 2: replace
A: that I will receive the investigation results within 24 hours -- that did not happen. The person
B: that I will receive the investigation results within XXXX hours -- that did not happen. The person

Difference 3: insert
A: but " customer closed the account ''. On XX/XX/XXXX, I physically went to Macy 's to
B: but " customer closed the account ''. On XX/XX/XXXX morning, I called Macy's/Citi bank phone number and attempted to pay the balance over the phone. After multiple transfer, a person, supposedly an employee of Macy 's, took my banking information ( account and routing numbers ) without paying the balance, and transferred me to a XXXX person of Macy 's that asked for the same payme

In [18]:
review = pd.read_csv(REVIEW_PATH, keep_default_na=False)

mask = review["review_id"].eq("ND001")
assert mask.sum() == 1

review.loc[mask, "review_label"] = "shared_template"
review.loc[mask, "review_notes"] = (
    "The narratives share extensive wording, but Text B adds a substantive "
    "account of a payment call, disclosure of banking details, and subsequent "
    "fraud-alert actions. This is more than a wording or redaction change. "
    "Shared wording does not establish identical underlying events."
)

review.to_csv(REVIEW_PATH, index=False)

print("ND001 annotation updated.")
print("Reviewed pairs:", review["review_label"].ne("").sum())

ND001 annotation updated.
Reviewed pairs: 30


In [19]:
def show_review_case(review_id, preview_chars=1500):
    """Display a review pair without revealing its similarity or issue labels."""
    matches = review.loc[review["review_id"].eq(review_id)]

    if len(matches) != 1:
        raise ValueError(f"Expected one review row for {review_id}")

    case = matches.iloc[0]

    for side, heading in [("left", "TEXT A"), ("right", "TEXT B")]:
        text = text_by_group.loc[case[f"{side}_text_group_id"]]

        print(f"\n{heading}:")
        print("Characters:", len(text))

        if preview_chars is None:
            print(text)
        else:
            print(text[:preview_chars])
            print("Preview truncated:", len(text) > preview_chars)

    print("\n" + "=" * 80)

In [20]:
pending_cases = review.loc[
    review["review_label"].eq("")
]

next_cases = (
    pending_cases
    .groupby("similarity_band", observed=True, sort=True)
    .head(1)
)

for review_id in next_cases["review_id"]:
    print("\nReview ID:", review_id)
    show_review_case(review_id)

In [21]:
show_review_case("ND002", preview_chars=None)


TEXT A:
Characters: 264
I've consistently ensured that payments on this account are made on time, never letting them become overdue. I'm uncertain about the reasons behind reports of late payment remarks on my accounts. According to 15 USC 1666b - Billing Error, this should be corrected.

TEXT B:
Characters: 255
I've consistently ensured that payments on this account are made on time, never letting them become overdue. I'm uncertain about the reasons for reports of late payment remarks on my accounts. According to 15 USC 1666b - Billing Error should be corrected.



In [22]:
review = pd.read_csv(REVIEW_PATH, keep_default_na=False)

annotations = {
    "ND002": {
        "label": "near_duplicate",
        "notes": (
            "Only minor phrasing and grammatical differences are present. "
            "The substantive statement and requested correction match."
        ),
    },
    "ND012": {
        "label": "shared_template",
        "notes": (
            "The narratives share extensive wording, but the stated balance "
            "changes from $0 to $2,700 and the account descriptor differs. "
            "These are material factual differences."
        ),
    },
    "ND022": {
        "label": "uncertain",
        "notes": (
            "The visible text strongly overlaps, but both previews are truncated. "
            "The remaining content must be checked before classification."
        ),
    },
}

assert review["review_id"].is_unique

for review_id, annotation in annotations.items():
    mask = review["review_id"].eq(review_id)
    assert mask.sum() == 1, f"Expected one row for {review_id}"

    review.loc[mask, "review_label"] = annotation["label"]
    review.loc[mask, "review_notes"] = annotation["notes"]

review.to_csv(REVIEW_PATH, index=False)

print("Annotated pairs:", review["review_label"].ne("").sum())
print("Uncertain pairs:", review["review_label"].eq("uncertain").sum())

Annotated pairs: 30
Uncertain pairs: 1


In [23]:
show_review_case("ND022", preview_chars=None)


TEXT A:
Characters: 1798
Dear Sir or Madam : I previously notified you that I am a victim of identity theft, and requested that you do the following : I still have not received written confirmation of these actions. As I stated before : I am a victim of identity theft. I recently learned that my personal information was used to open an account at your company. I did not open or authorize this account, and I request that it be closed immediately. Pleasesend me written confirmation that I am not responsible for charges on this account, and take appropriate steps to remove information about this account from my credit files. 
I have enclosed a copy of my FTCIdentity Theft Report. Because the information you are reporting is the result of identity theft, and inaccurate, I request that you stop reporting this information to the CRAs, as directed by section 623 ( a ) ( 1 ) ( B ) of the Fair Credit Reporting Act, 15 U.S.C. 1681s-2 ( a ) ( 1 ) ( B ). I ask that you take these steps as soon as

In [24]:
review = pd.read_csv(REVIEW_PATH, keep_default_na=False)

mask = review["review_id"].eq("ND022")
assert mask.sum() == 1

review.loc[mask, "review_label"] = "near_duplicate"
review.loc[mask, "review_notes"] = (
    "The full narratives share the same substantive wording and requested "
    "actions. Differences mainly concern redaction placeholders, formatting, "
    "and a minor wording change in the closing section. Redaction may conceal "
    "different entities; identical underlying events are not established."
)

review.to_csv(REVIEW_PATH, index=False)

print("Annotated pairs:", review["review_label"].ne("").sum())
print("Uncertain pairs:", review["review_label"].eq("uncertain").sum())

Annotated pairs: 30
Uncertain pairs: 0


In [25]:
pending_cases = review.loc[
    review["review_label"].eq("")
]

next_cases = (
    pending_cases
    .groupby("similarity_band", observed=True, sort=True)
    .head(1)
)

for review_id in next_cases["review_id"]:
    print("\nReview ID:", review_id)
    show_review_case(review_id, preview_chars=None)

In [26]:
review = pd.read_csv(REVIEW_PATH, keep_default_na=False)

annotations = {
    "ND003": {
        "label": "shared_template",
        "notes": (
            "The narratives share substantial wording, but Text A explicitly "
            "describes multiple denied applications while Text B describes "
            "one application with an application reference. The scope differs."
        ),
    },
    "ND013": {
        "label": "shared_template",
        "notes": (
            "The generic complaint wording matches, but one account descriptor "
            "names CITICARDS CBNA while the other is fully redacted. "
            "Equivalence of the referenced account cannot be established."
        ),
    },
    "ND023": {
        "label": "shared_template",
        "notes": (
            "The main wording matches, but Text A lists three card references "
            "and Text B lists two. The scope of referenced accounts may differ; "
            "this is not treated as a formatting-only change."
        ),
    },
}

assert review["review_id"].is_unique

for review_id, annotation in annotations.items():
    mask = review["review_id"].eq(review_id)
    assert mask.sum() == 1, f"Expected one row for {review_id}"

    review.loc[mask, "review_label"] = annotation["label"]
    review.loc[mask, "review_notes"] = annotation["notes"]

review.to_csv(REVIEW_PATH, index=False)

print("Annotated pairs:", review["review_label"].ne("").sum())
print("Uncertain pairs:", review["review_label"].eq("uncertain").sum())

Annotated pairs: 30
Uncertain pairs: 0


In [27]:
pending_cases = review.loc[
    review["review_label"].eq("")
]

next_cases = (
    pending_cases
    .groupby("similarity_band", observed=True, sort=True)
    .head(2)
)

for review_id in next_cases["review_id"]:
    print("\nReview ID:", review_id)
    show_review_case(review_id, preview_chars=None)

In [28]:
review = pd.read_csv(REVIEW_PATH, keep_default_na=False)

annotations = {
    "ND004": (
        "shared_template",
        "The generic wording matches, but the account descriptors name "
        "different institutions: CITICARDS CBNA and BRCLYSBANKDE.",
    ),
    "ND005": (
        "shared_template",
        "The narratives share extensive wording, but the requested recipient "
        "changes from Equifax to Experian. This is a material entity difference.",
    ),
    "ND014": (
        "shared_template",
        "The wording largely matches, but the institution referenced in the "
        "narrative changes from Navy Federal Credit Union to Capital One.",
    ),
    "ND015": (
        "shared_template",
        "The generic wording matches, but one account descriptor is fully "
        "redacted while the other names GOODLEAP LLC. Entity equivalence "
        "cannot be established.",
    ),
    "ND024": (
        "near_duplicate",
        "The visible substantive wording matches. Differences concern "
        "redaction-placeholder spacing and concatenation. Hidden entity "
        "differences cannot be ruled out.",
    ),
    "ND025": (
        "near_duplicate",
        "The substantive wording and requested action match. Differences "
        "mainly concern redaction placeholders, whitespace, and punctuation. "
        "Identical underlying events are not established.",
    ),
}

assert review["review_id"].is_unique

for review_id, (label, notes) in annotations.items():
    mask = review["review_id"].eq(review_id)
    assert mask.sum() == 1, f"Expected one row for {review_id}"

    review.loc[mask, "review_label"] = label
    review.loc[mask, "review_notes"] = notes

review.to_csv(REVIEW_PATH, index=False)

print("Annotated pairs:", review["review_label"].ne("").sum())
print("Uncertain pairs:", review["review_label"].eq("uncertain").sum())

Annotated pairs: 30
Uncertain pairs: 0


In [29]:
pending_cases = review.loc[
    review["review_label"].eq("")
]

next_cases = (
    pending_cases
    .groupby("similarity_band", observed=True, sort=True)
    .head(3)
)

for review_id in next_cases["review_id"]:
    print("\nReview ID:", review_id)
    show_review_case(review_id, preview_chars=None)

In [30]:
review = pd.read_csv(REVIEW_PATH, keep_default_na=False)

annotations = {
    "ND006": (
        "shared_template",
        "The institution changes from AFFIRM INC to TEKCOLLECT INC, "
        "and the stated balance changes from $270 to $1,500.",
    ),
    "ND007": (
        "near_duplicate",
        "The substantive wording and requested action match. Differences "
        "mainly concern institution-name abbreviation, phrasing, and spacing. "
        "Both texts retain a JPMCB CARD reference; entity identity is not "
        "independently verified.",
    ),
    "ND008": (
        "shared_template",
        "The institution changes, and Text B adds reference and balance "
        "information. The shared wording does not establish factual equivalence.",
    ),
    "ND016": (
        "shared_template",
        "The generic wording matches, but one text names Cap One and the "
        "other fully redacts the referenced entity. Equivalence is unresolved.",
    ),
    "ND017": (
        "shared_template",
        "The template wording matches, but one balance is redacted and the "
        "other is explicitly $490. Key factual equivalence cannot be established.",
    ),
    "ND018": (
        "near_duplicate",
        "The full detailed narrative and requested actions largely match. "
        "Differences mainly involve redaction substitutions, with no substantive "
        "added incident identified. Identical underlying events are not established.",
    ),
    "ND026": (
        "shared_template",
        "The shared narrative contains differences in account descriptors "
        "and the stated scope of late-payment references. These are not "
        "treated as formatting-only changes.",
    ),
    "ND027": (
        "shared_template",
        "The generic wording matches, but the texts explicitly reference "
        "different institutions: HARVARD COLLECTION and FIRSTPOINT.",
    ),
    "ND028": (
        "near_duplicate",
        "The visible substantive wording matches. Differences concern "
        "redaction-placeholder count and concatenation; hidden entity "
        "differences cannot be ruled out.",
    ),
}

assert review["review_id"].is_unique

for review_id, (label, notes) in annotations.items():
    mask = review["review_id"].eq(review_id)
    assert mask.sum() == 1, f"Expected one row for {review_id}"

    review.loc[mask, "review_label"] = label
    review.loc[mask, "review_notes"] = notes

review.to_csv(REVIEW_PATH, index=False)

print("Annotated pairs:", review["review_label"].ne("").sum())
print("Uncertain pairs:", review["review_label"].eq("uncertain").sum())

Annotated pairs: 30
Uncertain pairs: 0


In [31]:
pending_cases = review.loc[
    review["review_label"].eq("")
]

print("Remaining pairs:", len(pending_cases))

for review_id in pending_cases["review_id"]:
    print("\nReview ID:", review_id)
    show_review_case(review_id, preview_chars=None)

Remaining pairs: 0


In [32]:
review = pd.read_csv(REVIEW_PATH, keep_default_na=False)

annotations = {
    "ND009": (
        "shared_template",
        "The generic wording matches, but the account descriptors explicitly "
        "reference different institutions: TD BANK/TARGETCRED and DISCOVER BANK.",
    ),
    "ND010": (
        "near_duplicate",
        "The substantive statements and requested actions match. Differences "
        "mainly concern sentence order, grammar, and redaction of the form name. "
        "Identical underlying events are not established.",
    ),
    "ND019": (
        "shared_template",
        "The institution changes from American Express to Wells Fargo. "
        "Text B also adds substantive statements about discharged debt, "
        "reported balances, and a tax form.",
    ),
    "ND020": (
        "shared_template",
        "Text A retains an unfilled reporting-month template field, while "
        "Text B specifies a partially redacted date in 2023. Equivalence "
        "of the reporting period cannot be established.",
    ),
    "ND029": (
        "near_duplicate",
        "The visible substantive wording matches. The difference is limited "
        "to the length of the final redaction placeholder.",
    ),
    "ND030": (
        "near_duplicate",
        "The substantive wording and requested actions match. Differences "
        "mainly concern replacement of a possessive word with a redaction "
        "placeholder and whitespace.",
    ),
}

assert review["review_id"].is_unique

for review_id, (label, notes) in annotations.items():
    mask = review["review_id"].eq(review_id)
    assert mask.sum() == 1, f"Expected one row for {review_id}"

    review.loc[mask, "review_label"] = label
    review.loc[mask, "review_notes"] = notes

review.to_csv(REVIEW_PATH, index=False)

assert len(review) == 30
assert review["review_label"].ne("").all()
assert review["review_notes"].ne("").all()

print("All 30 review pairs have annotations.")

All 30 review pairs have annotations.


In [33]:
allowed_labels = {
    "near_duplicate",
    "shared_template",
    "same_topic",
    "uncertain",
}

assert review["review_label"].isin(allowed_labels).all()

review_summary = pd.crosstab(
    review["similarity_band"],
    review["review_label"],
).reindex(
    index=["0.90-0.95", "0.95-0.99", "0.99-1.00"],
    columns=[
        "near_duplicate",
        "shared_template",
        "same_topic",
        "uncertain",
    ],
    fill_value=0,
).fillna(0).astype(int)

review_summary["reviewed_pairs"] = review_summary.sum(axis=1)

display(review_summary)

review_summary.to_csv(
    PROJECT_ROOT / "reports/near_duplicate_review_summary.csv",
    index_label="similarity_band",
)

print("\nOverall annotation counts:")
print(review["review_label"].value_counts().to_string())

review_label,near_duplicate,shared_template,same_topic,uncertain,reviewed_pairs
similarity_band,,,,,
0.90-0.95,3,7,0,0,10
0.95-0.99,1,9,0,0,10
0.99-1.00,7,3,0,0,10



Overall annotation counts:
review_label
shared_template    19
near_duplicate     11


In [34]:
EVALUATION_DIR = PROJECT_ROOT / "evaluation"
EVALUATION_DIR.mkdir(parents=True, exist_ok=True)

review.to_csv(
    EVALUATION_DIR / "near_duplicate_review.csv",
    index=False,
)

print("Review annotations saved for version tracking.")

Review annotations saved for version tracking.


In [35]:
review_report = """# Near-Duplicate Candidate Review

## Method

The search uses 30,257 representatives of exact normalized text groups
from training-period credit card narratives.

Word unigram and bigram TF-IDF uses at most 50,000 features.
Each text retrieves up to five non-self cosine nearest neighbors.
Pairs with similarity at least 0.90 are retained and made unordered.

The search produced 2,942 candidate pairs involving 1,704 text groups.
Full search time was 45.29 seconds in the development environment.

## Review Design

Thirty candidate pairs were sampled: ten from each similarity band.
The random seed was 42.

Annotations were produced through a single human-guided,
assistant-assisted review process, not independent double annotation.

The shared_template category includes extensive shared wording with
material factual differences or unresolved entity/scope equivalence.
It does not establish a common author or a specific template source.

## Results

| Similarity band | Near duplicate | Shared template | Reviewed |
|---|---:|---:|---:|
| 0.90-0.95 | 3 | 7 | 10 |
| 0.95-0.99 | 1 | 9 | 10 |
| 0.99-1.00 | 7 | 3 | 10 |

Examples of important differences included institutions, balances,
application scope, and additional incident descriptions.

## Decision

Do not automatically delete or merge near-duplicate candidates.
Do not form transitive duplicate clusters solely from similarity scores.
Retain these records for the planned baseline, subject to the separate
exact-duplicate and training-label-conflict rules.

## Limitations

Similarity is a lexical feature-space measurement, not a probability.
Vocabulary filtering and redaction can conceal meaningful differences.
Top-five retrieval is not exhaustive.
Reviewed pairs may share text groups and are not independent observations.
The small stratified sample does not establish overall duplicate prevalence.
Recall cannot be estimated from reviewing retrieved candidates alone.
Text similarity does not establish identical underlying complaint events.

## Reproduction

Run notebooks/04_near_duplicate_review.ipynb from top to bottom.
Search settings are saved in configs/near_duplicate_vectorizer.json.
Reviewed pair metadata and notes are saved in
evaluation/near_duplicate_review.csv.
"""

(PROJECT_ROOT / "reports/near_duplicate_review.md").write_text(
    review_report,
    encoding="utf-8",
)

print("Near-duplicate review report saved.")

Near-duplicate review report saved.


In [36]:
POLICY_PATH = PROJECT_ROOT / "docs/training_sample_policy.md"
policy = POLICY_PATH.read_text(encoding="utf-8")

old_decision = "Finalize handling before fitting the baseline."
new_decision = (
    "Retain near-duplicate candidates for the baseline, subject to the "
    "separate exact-duplicate and training-label-conflict rules.\n"
    "Do not automatically merge candidates or form transitive clusters "
    "from similarity scores."
)

policy = policy.replace(old_decision, new_decision)

old_evaluation = (
    "Report full-period performance and a separate unseen-text evaluation."
)
new_evaluation = (
    "Report full-period performance and a separate subset excluding "
    "exact normalized text overlaps with training.\n"
    "Report near-duplicate similarities as diagnostics; do not claim "
    "that this subset is free of all text overlap."
)

policy = policy.replace(old_evaluation, new_evaluation)

POLICY_PATH.write_text(policy, encoding="utf-8")

print("Training sample policy updated.")

Training sample policy updated.
